# Day 4 — PP-OCRv4 rec (English) → INT8 TFLite
Pipeline: paddle → ONNX → onnxsim (fixed shape) → onnx2tf INT8 with calibration

In [ ]:
!pip install -q paddlepaddle paddle2onnx onnx onnxsim onnx2tf sng4onnx onnx-graphsurgeon ai_edge_litert tensorflow

In [ ]:
from google.colab import files
files.upload()  # pick ppocr_rec_en.zip

In [ ]:
!unzip -q -o ppocr_rec_en.zip
!ls ppocr_rec_en/

In [ ]:
# Paddle -> ONNX
!paddle2onnx --model_dir ppocr_rec_en \
    --model_filename inference.json \
    --params_filename inference.pdiparams \
    --save_file ppocr_rec_en.onnx \
    --opset_version 13
!ls -lh ppocr_rec_en.onnx

In [ ]:
# Fix dynamic shape -> static [1,3,48,320]
!python -m onnxsim ppocr_rec_en.onnx ppocr_rec_en_sim.onnx --overwrite-input-shape "x:1,3,48,320"
!ls -lh ppocr_rec_en_sim.onnx

In [ ]:
# Build calibration set (100 samples) — random for now; replace with real crops later
import numpy as np, os
os.makedirs('calib', exist_ok=True)
calib = np.random.rand(100, 1, 3, 48, 320).astype(np.float32)
np.save('calib/x.npy', calib)
print(calib.shape, calib.dtype)

In [ ]:
# onnx2tf with INT8 calibration
!onnx2tf -i ppocr_rec_en_sim.onnx -o saved_rec \
    -oiqt -qt per-tensor \
    -cind "x" "calib/x.npy" "[[[[0.0]]]]" "[[[[1.0]]]]"
!ls -lh saved_rec/ | grep tflite

In [ ]:
# Sanity check INT8 model
import tensorflow as tf, numpy as np
itp = tf.lite.Interpreter(model_path='saved_rec/ppocr_rec_en_sim_full_integer_quant.tflite')
itp.allocate_tensors()
inp, out = itp.get_input_details()[0], itp.get_output_details()[0]
print('in :', inp['shape'], inp['dtype'])
print('out:', out['shape'], out['dtype'])
x = np.random.randint(0, 255, inp['shape'], dtype=inp['dtype'])
itp.set_tensor(inp['index'], x); itp.invoke()
print('output shape:', itp.get_tensor(out['index']).shape)

In [ ]:
from google.colab import files
files.download('saved_rec/ppocr_rec_en_sim_full_integer_quant.tflite')